# Deity Analysis Pipeline

This notebook implements a 3-stage pipeline for analyzing deities in text:
1. **Translation Stage**: Detect language and translate non-English text
2. **Identification Stage**: Identify supernatural beings and entities
3. **Classification Stage**: Classify identified deities by gender and categories

The pipeline uses OpenAI's structured output features for reliable JSON responses.

## Setup and Configuration

In [ ]:
# Import required libraries
import csv
import os

import openai
import pandas as pd
from deity_analysis_utils import (
    process_classification_stage,
    process_identification_stage,
    process_translation_stage,
)

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

In [ ]:
# Configuration
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from config import DATA_INTERMEDIATE

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # set in .env (see .env.example)
# Input sample with regenerated "Text" column (licensed eHRAF text is not shipped; see README)
INPUT_CSV_PATH = f"{DATA_INTERMEDIATE}/paragraph_concat_sample.csv"
OUTPUT_DIR = f"{DATA_INTERMEDIATE}/pipeline_results/"

# Pipeline parameters
ENGLISH_THRESHOLD = 90.0  # Percentage of English content required to skip translation
MODEL = "gpt-4o-2024-08-06"  # OpenAI model to use
TEMPERATURE = 0  # Temperature for API calls
BATCH_SIZE = 10  # Number of rows to process before saving (optional)

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Initialize OpenAI client
client = openai.OpenAI(api_key=OPENAI_API_KEY)

## Load Data

In [ ]:
# Load input data
df = pd.read_csv(INPUT_CSV_PATH, encoding="utf-8-sig")
df = df.dropna(subset=["uuid"]).reset_index(drop=True)  # Remove rows with missing UUIDs
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
print(f"Loaded {len(df)} rows")
print(f"\nColumns: {df.columns.tolist()}")
print("\nFirst few rows:")
df.head()

## Stage 1: Translation

Detect the language of each text and translate if needed based on the English threshold.

In [ ]:
# Process translation stage
translation_df = process_translation_stage(df=df, text_column="Text", threshold=ENGLISH_THRESHOLD)

# Save translation results
translation_output_path = os.path.join(OUTPUT_DIR, "translation_results.csv")
translation_df = translation_df.dropna(subset=["uuid"]).reset_index(drop=True)

translation_df.to_csv(
    translation_output_path, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL
)

print(f"Translation stage complete. Results saved to {translation_output_path}")
print("\nTranslation Summary:")
print(f"Total rows: {len(translation_df)}")
print(f"Translated: {translation_df['was_translated'].sum()}")
print(f"Not translated: {(~translation_df['was_translated']).sum()}")
print("\nLanguage distribution:")
print(translation_df["detected_language"].value_counts())

In [ ]:
# Preview translation results
translation_df.head()
translation_df.to_csv(
    OUTPUT_DIR + "translation_results.csv", index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL
)

## Stage 2: Deity Identification

Identify supernatural beings, deities, and magical objects/forces in the translated text.

In [ ]:
# Process identification stage
identification_df = process_identification_stage(
    df=translation_df, client=client, text_column="translated_text", model=MODEL
)

# Save identification results
identification_output_path = os.path.join(OUTPUT_DIR, "identification_results.csv")
identification_df = identification_df.dropna(subset=["uuid"]).reset_index(drop=True)
identification_df.to_csv(
    identification_output_path, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL
)

print(f"Identification stage complete. Results saved to {identification_output_path}")
print("\nIdentification Summary:")
print(f"Total rows: {len(identification_df)}")
print(f"Rows with deities identified: {(identification_df['deities'].str.len() > 0).sum()}")
print(
    f"Total deities identified: {identification_df['deities'].apply(lambda x: len(x) if isinstance(x, list) else 0).sum()}"
)

In [ ]:
# Preview identification results
identification_df.head()

## Stage 3: Deity Classification

Classify identified deities by gender and various categories (creator, warrior, nature-related, etc.).

In [ ]:
# Process classification stage
classification_df = process_classification_stage(
    df=df,
    translation_df=translation_df,
    identification_df=identification_df,
    client=client,
    model=MODEL,
)

# Save classification results
classification_output_path = os.path.join(OUTPUT_DIR, "classification_results.csv")
classification_df.to_csv(classification_output_path, index=False, encoding="utf-8-sig")

print(f"Classification stage complete. Results saved to {classification_output_path}")

In [ ]:
# Preview classification results
classification_df.head()

## Combine All Results

In [ ]:
# Merge all stages into final dataframe
final_df = df.copy()
final_df = final_df.merge(translation_df, on="uuid", how="left")
final_df = final_df.merge(identification_df, on="uuid", how="left")
final_df = final_df.merge(classification_df, on="uuid", how="left")

# Save complete results
final_output_path = os.path.join(OUTPUT_DIR, "deity_analysis_complete.csv")
final_df.to_csv(final_output_path, index=False, encoding="utf-8-sig")

print("\nAll stages complete!")
print(f"Final results saved to {final_output_path}")
print(f"\nFinal dataframe shape: {final_df.shape}")

In [ ]:
# Display final results summary
print("=" * 80)
print("PIPELINE SUMMARY")
print("=" * 80)
print(f"\nTotal texts processed: {len(final_df)}")
print(
    f"Texts translated: {final_df['was_translated'].sum() if 'was_translated' in final_df else 'N/A'}"
)
print(
    f"Texts with deities identified: {(final_df['deities'].str.len() > 0).sum() if 'deities' in final_df else 'N/A'}"
)
print(
    f"Total deities identified: {final_df['deities'].apply(lambda x: len(x) if isinstance(x, list) else 0).sum() if 'deities' in final_df else 'N/A'}"
)

# Gender distribution
if "gender" in final_df.columns:
    all_genders = []
    for genders in final_df["gender"]:
        if isinstance(genders, list):
            all_genders.extend(genders)
    print("\nGender distribution:")
    gender_counts = pd.Series(all_genders).value_counts()
    print(gender_counts)

# Category distribution (top 10)
cat_cols = [col for col in final_df.columns if col.startswith("cat_") and col != "cat_type"]
if cat_cols:
    print("\nTop 10 categories by frequency:")
    cat_sums = {}
    for col in cat_cols:
        total = sum([sum(x) if isinstance(x, list) else 0 for x in final_df[col]])
        cat_sums[col] = total
    top_cats = sorted(cat_sums.items(), key=lambda x: x[1], reverse=True)[:10]
    for cat, count in top_cats:
        print(f"  {cat}: {count}")

In [ ]:
# Preview final results
final_df.head()